# LAB- P7: El Modelo de Equilibrio General Dinámico Básico (DGE) (Julia)
**Proyecto MACRO-AI-COMP (Convocatoria INNOVA26, UMA / Banco Santander)**
*   **Código de Práctica**: LAB-P7-v1.0
*   **Capítulo de Referencia**: Capítulo 8, *An Introduction to Computational Macroeconomics* (Bongers, Gómez y Torres, Vernon Press, 2019)
*   **Autores**: Equipo Docente MACRO-AI-COMP
*   **Objetivo**: Desarrollar e interactuar con el modelo canónico de Equilibrio General Dinámico (DGE) en tiempo discreto, resolviendo y comparando la aproximación linealizada de Blanchard-Khan frente a la solución numérica no lineal exacta.

---

## Objetivos de Aprendizaje
Al finalizar esta práctica, serás capaz de:
1.  **Comprender** la microfundamentación intertemporal de las decisiones de consumo y ahorro y su interacción con el sector productivo en equilibrio general.
2.  **Aplicar** el método de Blanchard-Khan para resolver y simular sistemas dinámicos lineales con expectativas racionales y variables forward-looking.
3.  **Analizar** el mecanismo de propagación intertemporal y la persistencia de shocks tecnológicos (TFP) sobre las variables macroeconómicas clave (Producción, Consumo, Inversión, Capital y Tasa de Interés).
4.  **Evaluar** la precisión y los límites de la aproximación linealizada frente a la solución no lineal exacta ante perturbaciones de distinta magnitud.
5.  **Identificar** discrepancias de programación y timing en modelos macroeconómicos computacionales complejos.


## 1. El Modelo Canónico de Equilibrio General Dinámico (DGE)

Los modelos de Equilibrio General Dinámico (DGE), en sus versiones deterministas o estocásticas (DSGE), constituyen el núcleo de la macroeconomía moderna. A diferencia de los modelos IS-LM, los modelos DGE se basan en la **microfundamentación**: las ecuaciones de comportamiento se derivan directamente de los problemas de optimización intertemporal de los consumidores y las empresas.

### 1.1 Estructura del Modelo
El modelo canónico desarrollado en el Capítulo 8 representa una economía cerrada y perfectamente competitiva sin elecciones de ocio ($L_t = 1$):

1. **Problema del Consumidor:**
   Maximizar la utilidad descontada intertemporal:
   $$\max_{\{C_t\}} \sum_{t=0}^{\infty} \beta^t \ln(C_t)$$
   Sujeto a la restricción presupuestaria intertemporal:
   $$C_t + K_{t+1} = W_t + (R_t + 1 - \delta) K_t$$

2. **Problema de la Empresa:**
   Maximizar beneficios periodo a periodo:
   $$\max_{\{K_t, L_t\}} Y_t - W_t L_t - R_t K_t$$
   Sujeta a la función de producción Cobb-Douglas:
   $$Y_t = A_t K_t^\alpha L_t^{1-\alpha}$$

3. **Condición de Equilibrio (Vaciado de Mercados):**
   $$Y_t = C_t + I_t \quad \text{y} \quad K_{t+1} = (1-\delta) K_t + I_t$$

### 1.2 Sistema de Ecuaciones Reducido
El equilibrio competitivo dinámico de esta economía se reduce a un sistema de dos ecuaciones diferenciales en diferencias (una para la variable rígida/predeterminada $K_t$ y otra para la variable de expectativas/flexible $C_t$), alimentado por el proceso autorregresivo de la productividad (TFP):
1. **Dinámica del Capital:**
   $$K_{t+1} = (1-\delta)K_t + A_t K_t^\alpha - C_t$$
2. **Ecuación de Euler (Keynes-Ramsey):**
   $$C_{t+1} = \beta [ \alpha A_{t+1} K_{t+1}^{\alpha-1} + 1 - \delta ] C_t$$
3. **Proceso de la TFP:**
   $$\ln(A_t) = \rho \ln(A_{t-1}) + \epsilon_t$$
   Donde $\epsilon_t$ es un shock tecnológico exógeno y $\rho$ es la persistencia.

### 1.3 Estado Estacionario
Estableciendo $A = 1$ y eliminando los subíndices de tiempo en el sistema, obtenemos los valores de largo plazo:
$$\bar{R} = \frac{1 - \beta + \beta\delta}{\beta}, \quad \bar{K} = \left( \frac{\bar{R}}{\alpha} \right)^{\frac{1}{\alpha - 1}}, \quad \bar{Y} = \bar{K}^\alpha, \quad \bar{I} = \delta \bar{K}, \quad \bar{C} = \bar{Y} - \bar{I}$$


In [ ]:
# Este cuaderno depende del paquete `MacroAIComp` (Project.toml/Manifest.toml
# en la raíz del repositorio). En MyBinder (ver docs/DESPLIEGUE_BINDER.md) y en
# tu entorno local, el kernel ya arranca dentro del repositorio clonado, así
# que la celda siguiente activa e instancia el proyecto automáticamente.
# Nota: Google Colab no soporta Julia de forma nativa desde un notebook .ipynb;
# para la versión Julia de esta práctica usa MyBinder.


In [ ]:
using Pkg
Pkg.activate("../..")
Pkg.instantiate()

using MacroAIComp
using Plots
import Plots: mm
default(gridalpha=0.6, gridstyle=:dot)  # estilo de grid consistente con la versión Python
using LinearAlgebra
using NLsolve
using Interact
using BenchmarkTools


## 2. Simulación Interactiva: Shock Tecnológico Transitorio (TFP)

Imaginemos que la economía parte de su estado estacionario. En el periodo $t=1$, se produce un shock tecnológico transitorio positivo del $1\%$ ($\epsilon_1 = 0.01$) en la TFP, que decae de acuerdo con la persistencia $\rho = 0.8$.

Dado que la productividad aumenta, la economía experimenta un incremento inmediato en la capacidad de producción. Los consumidores, al tener expectativas de que la productividad y la tasa de interés serán más altas, aumentan su consumo inmediatamente ($C_1$ salta). Sin embargo, al ser el capital un factor rígido, no puede saltar en el periodo del shock ($K_1 = \bar{K}$). El aumento de producción se destina tanto a consumo como a inversión, lo que genera una acumulación gradual de capital físico. Este capital alcanzará su pico varios periodos después (efecto *hump-shape* o respuesta jorobada) antes de converger lentamente hacia el equilibrio inicial.


In [ ]:
params_base = default_calibration(DGEParams)
ss = compute_steady_state(params_base)

println("VALORES DE EQUILIBRIO (SS):")
println("  Capital (K*)   : ", round(ss["K"], digits=4))
println("  Consumo (C*)   : ", round(ss["C"], digits=4))
println("  Producción (Y*): ", round(ss["Y"], digits=4))
println("  Inversión (I*) : ", round(ss["I"], digits=4))


In [ ]:
# Verificación de los valores del estado estacionario contra el oráculo
# (Tabla 8.2 del libro, reproducido en oraculo.md).
# La calibración por defecto de DGEParams es alpha=0.35, beta=0.96, delta=0.06, A=1.0,
# que coincide con la calibración base del oráculo.

@assert isapprox(ss["K"], 6.698596; atol=1e-6)
@assert isapprox(ss["Y"], 1.945783; atol=1e-6)
@assert isapprox(ss["C"], 1.543867; atol=1e-6)
@assert isapprox(ss["I"], 0.401916; atol=1e-6)
@assert isapprox(ss["R"], 0.10166666666666667; atol=1e-6)
println("OK: estado estacionario coincide con el oráculo (Apéndice N).")


## 2.1 Verificación frente al oráculo

Comparamos contra los valores reportados en el libro (Tabla 8.2) y reproducidos por el
código MATLAB/DYNARE del Apéndice N, recogidos en `oraculo.md`:

**Estado estacionario (calibración base: α=0.35, β=0.96, δ=0.06, A=1.0):**

| Magnitud | Valor esperado (oráculo) |
|---|---|
| Capital en SS (K*) | 6.698596 |
| Producción en SS (Y*) | 1.945783 |
| Consumo en SS (C*) | 1.543867 |
| Inversión en SS (I*) | 0.401916 |
| Tipo de interés en SS (R*) | 0.10166666666666667 |

**Blanchard-Khan — Estabilidad:**

| Magnitud | Valor esperado (oráculo) |
|---|---|
| Autovalor estable μ₁ (|μ| < 1) | ≈0.90399 |
| Autovalor inestable μ₂ (|μ| > 1) | ≈1.15229 |
| Clasificación | Punto de silla (exactamente 1 raíz estable) |

**Shock temporal de PTF (+1% con ρ=0.8):**

| Magnitud | Valor esperado (oráculo) |
|---|---|
| K₁ (predeterminado, no salta) | = K* ≈ 6.6986 |
| C₁ (salta al alza en impacto) | > C* |
| Y₁ (salta al alza en impacto) | > Y* |
| I₁ (salta al alza en impacto) | > I* |
| Pico de K (hump, ocurre con retardo) | Entre periodos 2 y 12 |
| Convergencia de largo plazo (C, K) | Vuelven al SS inicial (tol 1e-3) |
| Consistencia Blanchard-Khan vs simulación no lineal | K y C coinciden con rtol 1e-2 |

Así puedes comparar a simple vista, sin abrir `oraculo.md`, el número que
debería salir en cada celda siguiente con el que realmente sale.


In [ ]:
# Simulación interactiva: Shock Tecnológico Transitorio
@manipulate for epsilon in slider(-0.05:0.005:0.05; value=0.01, label="Shock (ε1)"), rho_val in slider(0.0:0.05:0.99; value=0.80, label="Persist. (ρ)"), alpha_val in slider(0.2:0.05:0.5; value=0.35, label="Elasticidad (α)"), beta_val in slider(0.90:0.01:0.99; value=0.96, label="Descuento (β)"), delta_val in slider(0.01:0.01:0.15; value=0.06, label="Deprec. (δ)")

    params = DGEParams(alpha_val, beta_val, delta_val, rho_val, 1.0)
    ss_sim = compute_steady_state(params)
    K0 = ss_sim["K"]
    T_sim = 50
    
    # Path del TFP: shock ocurre en t=1 (índice 2 en Julia), igual que en Python
    A_path = ones(T_sim)
    a_shock = zeros(T_sim)
    a_shock[2] = epsilon
    for t in 3:T_sim
        a_shock[t] = rho_val * a_shock[t-1]
    end
    A_path .= exp.(a_shock)

    res = solve_nonlinear_simulation(params, K0, A_path, T_sim)

    t_axis = 0:(T_sim - 1)
    t_shock = 1

    p1 = plot(t_axis, res["Y"], color="#004C97", lw=2.5, label="Producción (Y)")
    hline!([ss_sim["Y"]], color=:gray, ls=:dot, label="")
    vline!([t_shock], color=:grey, ls=:dot, alpha=0.7, label="")
    title!("Evolución de la Producción (PIB)")
    xlabel!("Período (t)")
    ylabel!("Y")

    p2 = plot(t_axis, res["C"], color="#7A3E9F", lw=2.5, label="Consumo (C)")
    hline!([ss_sim["C"]], color=:gray, ls=:dot, label="")
    vline!([t_shock], color=:grey, ls=:dot, alpha=0.7, label="")
    title!("Evolución del Consumo Privado")
    xlabel!("Período (t)")
    ylabel!("C")

    p3 = plot(t_axis, res["I"], color="#D95319", lw=2.5, label="Inversión (I)")
    hline!([ss_sim["I"]], color=:gray, ls=:dot, label="")
    vline!([t_shock], color=:grey, ls=:dot, alpha=0.7, label="")
    title!("Dinámica de Inversión")
    xlabel!("Período (t)")
    ylabel!("I")

    p4 = plot(t_axis, res["K"], color="#8EAD3A", lw=2.5, label="Capital (K)")
    hline!([ss_sim["K"]], color=:gray, ls=:dot, label="")
    vline!([t_shock], color=:grey, ls=:dot, alpha=0.7, label="")
    title!("Trayectoria de Acumulación de Capital")
    xlabel!("Período (t)")
    ylabel!("K")
    
    plot(p1, p2, p3, p4, layout=(2,2), size=(900, 600), 
         plot_title="Ajuste Dinámico frente a Shock TFP (No Lineal)", top_margin=10mm)
end


## 2.1 Verificación frente al oraculo

Comparamos contra los valores reportados en el libro (Tabla 8.2) y reproducidos por el
codigo MATLAB/DYNARE del Apendice N, recogidos en `oraculo.md`:

**Estado estacionario (calibracion base: alpha=0.35, beta=0.96, delta=0.06, A=1.0):**

| Magnitud | Valor esperado (oraculo) |
|---|---|
| Capital en SS (K*) | 6.698596 |
| Produccion en SS (Y*) | 1.945783 |
| Consumo en SS (C*) | 1.543867 |
| Inversion en SS (I*) | 0.401916 |
| Tipo de interes en SS (R*) | 0.10166666666666667 |

**Blanchard-Khan — Estabilidad:**

| Magnitud | Valor esperado (oraculo) |
|---|---|
| Autovalor estable mu1 (|mu| < 1) | approx 0.90399 |
| Autovalor inestable mu2 (|mu| > 1) | approx 1.15229 |
| Clasificacion | Punto de silla (exactamente 1 raiz estable) |

**Shock temporal de PTF (+1% con rho=0.8):**

| Magnitud | Valor esperado (oraculo) |
|---|---|
| K1 (predeterminado, no salta) | = K* approx 6.6986 |
| C1 (salta al alza en impacto) | > C* |
| Y1 (salta al alza en impacto) | > Y* |
| I1 (salta al alza en impacto) | > I* |
| Pico de K (hump, ocurre con retardo) | Entre periodos 2 y 12 |
| Convergencia de largo plazo (C, K) | Vuelven al SS inicial (tol 1e-3) |
| Consistencia Blanchard-Khan vs simulacion no lineal | K y C coinciden con rtol 1e-2 |


## 3. Límites de la Aproximación Lineal: Blanchard-Khan frente a Modelo No Lineal

El método de Blanchard-Khan (o cualquier método de linealización de primer orden) asume que la economía realiza fluctuaciones pequeñas en torno a su estado estacionario, de modo que la curvatura de las funciones de utilidad y producción puede aproximarse mediante tangencias de primer orden.

Sin embargo, si la economía sufre perturbaciones de gran magnitud (por ejemplo, una caída catastrófica de TFP o un shock tecnológico masivo), la aproximación lineal comete un **error de truncamiento** sistemático.

A continuación, puedes simular shocks de diferente envergadura y evaluar visualmente cómo difieren las soluciones calculadas mediante:
1. **Blanchard-Khan (Log-Linealizado)**: Resuelve el sistema linealizado mediante la descomposición de autovalores de la matriz $J$.
2. **NLsolve.jl (No Lineal)**: Resuelve el sistema completo de ecuaciones no lineales en niveles mediante métodos iterativos de Newton.


In [ ]:
# Comparación Lineal (Blanchard-Kahn) vs No Lineal
@manipulate for epsilon_shock in 0.01:0.01:0.10
    
    params = default_calibration(DGEParams)
    ss_comp = compute_steady_state(params)
    K0 = ss_comp["K"]
    T_sim = 40
    
    A_path = ones(T_sim)
    a_s = zeros(T_sim)
    a_s[2] = epsilon_shock  # shock ocurre en t=1 (índice 2 en Julia), igual que en Python
    for t in 3:T_sim
        a_s[t] = params.rho * a_s[t-1]
    end
    A_path .= exp.(a_s)

    # Resolver
    res_lin = solve_blanchard_khan(params, K0, A_path, T_sim)
    res_nonlin = solve_nonlinear_simulation(params, K0, A_path, T_sim)

    t_axis = 0:(T_sim - 1)
    t_shock = 1

    # Error relativo máximo en Consumo y Capital
    diff_C = maximum(abs.(res_lin["C"] .- res_nonlin["C"])) / ss_comp["C"] * 100
    diff_K = maximum(abs.(res_lin["K"] .- res_nonlin["K"])) / ss_comp["K"] * 100
    println("Error relativo máximo en Consumo : ", round(diff_C, digits=4), "%")
    println("Error relativo máximo en Capital  : ", round(diff_K, digits=4), "%")

    p1 = plot(t_axis, res_lin["C"], color="#7A3E9F", ls=:dash, lw=2, label="Blanchard-Khan (Linealizado)")
    plot!(t_axis, res_nonlin["C"], color="#7A3E9F", lw=2, label="Exacto No Lineal")
    vline!([t_shock], color=:grey, ls=:dot, alpha=0.7, label="")
    title!("Consumo (C): Comparación de resolvedores")
    xlabel!("Periodo (t)")
    ylabel!("C")

    p2 = plot(t_axis, res_lin["K"], color="#8EAD3A", ls=:dash, lw=2, label="Blanchard-Khan (Linealizado)")
    plot!(t_axis, res_nonlin["K"], color="#8EAD3A", lw=2, label="Exacto No Lineal")
    vline!([t_shock], color=:grey, ls=:dot, alpha=0.7, label="")
    title!("Capital (K): Comparación de resolvedores")
    xlabel!("Periodo (t)")
    ylabel!("K")

    plot(p1, p2, layout=(1,2), size=(900, 400), top_margin=10mm)
end


In [ ]:
# Verificacion de los autovalores de Blanchard-Khan y la consistencia
# entre la solucion linealizada y la no lineal (oraculo.md, Apendice N).

# --- Autovalores BK ---
# Replicamos el calculo de la matriz J del sistema linealizado.
alpha_v = params_base.alpha
beta_v = params_base.beta
delta_v = params_base.delta

Omega_val = 1.0 - beta_v + beta_v * delta_v
Phi_val = 1.0 - beta_v + (1.0 - alpha_v) * beta_v * delta_v

A_mat = [1.0 0.0; Omega_val -alpha_v * beta_v * delta_v]
B_mat = [0.0 alpha_v; Phi_val 0.0]
D_mat = [1.0 Omega_val; 0.0 1.0]
F_mat = [-Omega_val 0.0; 0.0 0.0]
G_mat = [1.0 0.0; 0.0 1.0 - delta_v]
H_mat = [0.0 0.0; 0.0 delta_v]

invA = inv(A_mat)
inv_term = inv(D_mat + F_mat * invA * B_mat)
J = inv_term * (G_mat + H_mat * invA * B_mat)

mu_vals = real(eigvals(J))
mu_sorted = sort(abs.(mu_vals))
mu_s = mu_sorted[1]  # estable
mu_u = mu_sorted[2]  # inestable

@assert isapprox(mu_s, 0.90399; atol=1e-5)
@assert isapprox(mu_u, 1.15229; atol=1e-5)
println("OK: autovalores BK coinciden con el oraculo (Apendice N).")

# --- Consistencia lineal vs. no lineal (shock +1% con rho=0.8) ---
T_check = 60
a_hat_check = zeros(T_check)
a_hat_check[2] = 0.01  # shock en t=1 (indice 2 en Julia)
for t in 3:T_check
    a_hat_check[t] = params_base.rho * a_hat_check[t-1]
end
A_path_check = exp.(a_hat_check)

res_bk = solve_blanchard_khan(params_base, ss["K"], A_path_check, T_check)
res_nl = solve_nonlinear_simulation(params_base, ss["K"], A_path_check, T_check)

# Verificar que K y C de ambas soluciones coinciden (rtol 1e-2)
for key in ["K", "C"]
    max_diff = maximum(abs.(res_bk[key] .- res_nl[key]))
    rel_diff = max_diff / ss[key]
    @assert rel_diff < 0.02 "La discrepancia relativa en $key ($rel_diff) supera rtol=1e-2"
end
println("OK: soluciones BK y no lineal coinciden con rtol=1e-2 (oraculo).")

# --- Convergencia de largo plazo al SS inicial ---
@assert isapprox(res_nl["K"][end], ss["K"]; atol=1e-3)
@assert isapprox(res_nl["C"][end], ss["C"]; atol=1e-3)
println("OK: convergencia de largo plazo al SS inicial (tol 1e-3, oraculo).")


## 4. Cuaderno de Bitácora (Actividades para el Alumno)

Responde a las siguientes cuestiones tras interactuar con el modelo de equilibrio general dinámico:

1.  **Mecanismo de Propagación del Shock**:
    *   Establece un shock tecnológico $\epsilon_1 = 0.01$ y una persistencia $\rho = 0.8$. Describe el patrón dinámico de las 4 variables graficadas en la simulación 1. ¿Por qué el consumo y la inversión saltan instantáneamente en $t=1$, mientras que el stock de capital responde con retraso?
    *   Reduce la persistencia a $\rho = 0.2$. ¿Cómo cambia la trayectoria de consumo y capital? Explica la relación entre la persistencia del shock tecnológico y la velocidad de acumulación de capital.
2.  **Límites de la Log-Linealización**:
    *   En la simulación 2, establece un shock de TFP de $\epsilon = 0.01$ (un shock estándar de $1\%$). Observa el error relativo máximo entre Blanchard-Khan y el resolvedor no lineal exacto. ¿Qué magnitud tiene?
    *   Incrementa el shock tecnológico a $\epsilon = 0.25$ (un shock masivo de productividad del $25\%$). ¿Qué ocurre con el error relativo en consumo y capital? Explica por qué la aproximación de primer orden de Blanchard-Khan pierde validez cuando las desviaciones respecto al estado estacionario son grandes.
3.  **Higiene IA y Validación cruzada**:
    *   Verifica que los valores de estado estacionario calculados por Python coincidan con la Tabla 8.2 del libro impreso ($\bar{K}=6.699$, $\bar{Y}=1.946$, $\bar{C}=1.544$, $\bar{R}=0.102$). Reporta el resultado en tu bitácora.


## 5. Buenas Prácticas Aplicadas en este Laboratorio
1.  **Aislamiento Paramétrico**: El código de las rutinas matriciales y resolvedores de Blanchard-Khan y Newton no lineal están desacoplados en el módulo [dge.jl](file:///c:/Users/AntonioRC/Desktop/PIE/src/macroaicomp/models/dge.jl), importados de manera modular.
2.  **Corrección de Timing y Parámetros**: Hemos solucionado la discrepancia de indexación de TFP del código MATLAB impreso, ofreciendo una opción `use_matlab_timing` para que los estudiantes comprueben la equivalencia con la hoja del libro y con el modelo de Dynare correcto.
3.  **Higiene del Repositorio**: El cuaderno se somete automáticamente a `nbstripout` mediante hooks de `pre-commit` para evitar subir gráficos cargados y metadatos volátiles de Jupyter al control de versiones.


## 6. Benchmark de Rendimiento (Fase III)
Evaluamos la velocidad de simulación usando `BenchmarkTools.jl`.

In [ ]:
# Benchmark simulation para No Lineal y Blanchard-Kahn
A_bench = ones(50)
A_bench[1] = exp(0.01)
a_bench = zeros(50); a_bench[1] = 0.01

println("Benchmark NLsolve (No Lineal):")
@btime solve_nonlinear_simulation($params_base, $ss["K"], $A_bench, 50)

println("Benchmark Blanchard-Kahn (Lineal):")
@btime solve_blanchard_khan($params_base, $ss["K"], $A_bench, 50)
